In [1]:
import numpy as np
import pandas as pd

C:\Users\LJENG\AppData\Roaming\Python\Python38\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


file name -> Columns

quater-i.csv -> ['order_id', 'quantity', 'item_id', 'choice_description_id' 'item_price']

items.csv -> ['item_id', 'item_name']

In [2]:
# import like this
items_path = "items.csv"
q1_path = "quarter-1.csv"
q2_path = "quarter-2.csv"
q3_path = "quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

### `Q:1-5`
1. You are given three quater files, your job is to append these three files and make a single dataframe.
2. Have a index as Q-1 Q-2 Q-3 for respective quater files in the dataframe
3. Your are given a file items.csv which has item_id and item_name. Find out most sold items in each quarter.
4. Find out items which has made most revenue in each quarter.
5. Find out avg order price of each quarter.

***Note: item_price is given as str with $ sign, in earlier task you have converted this to rupees, here too first convert item_price field in rupees.***

In [3]:
# Q1
df=pd.concat((q1,q2,q3))
df

,order_id,quantity,item_id,choice_description_id,item_price
0,1,1,1,1,$3.39
1,1,1,2,2,$3.39
2,2,2,4,3,$16.98
3,4,1,7,6,$9.25
4,6,1,9,8,$8.75
...,...,...,...,...,...
2342,1829,1,23,92,$11.25
2343,1830,1,23,1043,$11.25
2344,1832,1,10,116,$8.75
2345,1832,1,8,0,$4.45


In [4]:
# Q2
df=pd.concat((q1,q2,q3),keys=["Q-1","Q-2","Q-3"])
df

order_id quantity item_id choice_description_id item_price
Q-1 0           1        1       1                     1     $3.39 
    1           1        1       2                     2     $3.39 
    2           2        2       4                     3    $16.98 
    3           4        1       7                     6     $9.25 
    4           6        1       9                     8     $8.75 
...           ...      ...     ...                   ...        ...
Q-2 2342     1829        1      23                    92    $11.25 
    2343     1830        1      23                  1043    $11.25 
    2344     1832        1      10                   116     $8.75 
    2345     1832        1       8                     0     $4.45 
    2346     1834        1      20                   515    $11.25 

[4622 rows x 5 columns]

In [5]:
# Q3
df=df.reset_index()
df
df=df.merge(items,on="item_id")

<!--  -->

In [6]:
df.groupby(["level_0","item_id"],as_index=False)["quantity"].sum().max()

level_0     Q-2
item_id      49
quantity    394
dtype: object

In [7]:
df.groupby(["level_0","item_id"],as_index=False)["quantity"].sum().sort_values("quantity",ascending=False)
# Q3 is empty

,level_0,item_id,quantity
53,Q-2,4,394
4,Q-1,4,367
60,Q-2,11,304
11,Q-1,11,287
8,Q-1,8,258
...,...,...,...
93,Q-2,47,1
48,Q-1,49,1
86,Q-2,37,1
42,Q-1,42,1


In [13]:
#Q4
df["price"]=df.item_price.apply(lambda x:float(x[1:]))
df["revenue"]=df["quantity"]*df["price"]
df

,level_0,level_1,order_id,quantity,item_id,choice_description_id,item_price,item_name,price,revenue
0,Q-1,0,1,1,1,1,$3.39,Izze,3.39,3.39
1,Q-1,10,12,1,1,18,$3.39,Izze,3.39,3.39
2,Q-1,25,21,1,1,33,$3.39,Izze,3.39,3.39
3,Q-1,34,30,1,1,33,$3.39,Izze,3.39,3.39
4,Q-1,179,155,1,1,33,$3.39,Izze,3.39,3.39
...,...,...,...,...,...,...,...,...,...,...
4617,Q-2,1411,1094,1,48,755,$8.49,Veggie Salad,8.49,8.49
4618,Q-2,1528,1192,1,48,149,$8.49,Veggie Salad,8.49,8.49
4619,Q-2,1605,1263,1,48,306,$8.49,Veggie Salad,8.49,8.49
4620,Q-2,1745,1395,1,48,755,$8.49,Veggie Salad,8.49,8.49


In [30]:
df.groupby(["level_0","item_name"],as_index=False)["revenue"].sum().sort_values("revenue",ascending=False).drop_duplicates("level_0")

,level_0,item_name,revenue
65,Q-2,Chicken Bowl,4192.25
17,Q-1,Chicken Bowl,3852.38


In [42]:
df.groupby(["order_id","level_0"],as_index=False)["price"].sum().groupby("level_0")["price"].mean()

level_0
Q-1    11.929405
Q-2    11.888547
Name: price, dtype: float64

### `Q-6` From the IPL wala dataset you have to find the Purple cap holder each season.

*Note: Bowler with most no wickets in a season gets purple cap. If more than one bowler have same no of wickets in the season, one with least ecomnomy among them is purple cap holder.*

Bowler's Economy = runs-conceded per six ballsum

In [10]:
balls = pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
matches = pd.read_csv("IPL_Matches_2008_2022.csv")

Best bowler in death overs.
Note: Have taken most no of wickets in case of tie with least economy

Death Overs - [16-20]

### `Q-8` Batsman record season wise

Make a function which takes a input `batsman_name` and it returns a dataframe.
Columns of the data frame are - `['Season','Innings', 'TotalRuns', 'Avg', 'HighestScore','StrikeRate']`.
* In result make `Season` column as index.

* Avg - total_runs/ no of time got out. - player_out column will help.
* StrikeRate -(total_runs/ balls faced) * 100- wides are not included in batsman ball faced counts. No balls are included. -> Extra_type column will help
* Batsman Can score runs on No Balls.
* Batsman can get out on No Ball or Wides. And even while being on non-striker. Keep these things in mind before masking.

In [11]:
# code here
batterdf = seasondf.copy()
batterdf["IsBatsmanBall"] = batterdf["extra_type"].apply(lambda x: 1 if x != "wides" else 0)

def bat_record_season(batsman):
    bdf = batterdf[batterdf.batter == batsman].copy()
    bdf["IsBatsmanOut"] = bdf.batter == bdf.player_out
    df = bdf.groupby(["Season", "ID"], as_index = False)[["batsman_run", "IsBatsmanBall", "IsBatsmanOut"]].sum()
    innings = df.groupby("Season").ID.count()
    df = df.groupby("Season").agg(
    {
        "batsman_run":["sum", "max"],
        "IsBatsmanBall": "sum",
        "IsBatsmanOut": "sum"
    })
    df["Innings"]= innings
    df["TotalRuns"] = df[("batsman_run", "sum")]
    df["Avg"] = df["TotalRuns"] / df[("IsBatsmanOut", "sum")]
    df["HighestScore"] = df[("batsman_run", "max")]
    df["StrikeRate"] = df["TotalRuns"] / df[("IsBatsmanBall", "sum")] * 100
    return df.drop(columns = ["batsman_run", "IsBatsmanBall", "IsBatsmanOut"])
bat_record_season("MS Dhoni")

NameError: name 'seasondf' is not defined

### `Q-9` Using both dataset, make a dataframe as described below

Data Frame columns-> `['PlayerOfThematch', 'BattingFigure', 'BowlingFigure']`

* BattingFigure->`<runs>/<balls>`
* BowlingFigure->`<wicket>/<runs-conceded>`

DataFrame should have one record for each match.

Say 'V Kohli' got POM award then in dataset include his batting figure of that match. Say he scored 112runs in 76 balls. And he hasn't bowled so Bowling Figure will be NaN
```
PlayerOfThematch BattingFigure BowlingFigure
V Kohli          112/76         nan  

```


In [ ]:
# code here
df = balls.merge(matches[["ID", "Player_of_Match"]], on="ID")
batterdf = df[df.batter == df.Player_of_Match].copy()
batterdf["IsBatsmanBall"] = batterdf["extra_type"].apply(lambda x: 1 if x != "wides" else 0)
batter = batterdf.groupby(["ID", "batter"], as_index=False)[["batsman_run", "IsBatsmanBall"]].sum()
batter["BattingFigure"] = batter[["batsman_run", "IsBatsmanBall"]].apply(lambda x: '/'.join(map(str, x.values)), axis = 1)
batter.rename(columns = {"batter": "PlayerOfMatch"}, inplace = True)
batter.head()

In [ ]:
bowlerdf = df[df.bowler == df.Player_of_Match].copy()
bowlerdf["IsBowlerWicket"] = bowlerdf["kind"].apply(lambda x: 1 if x in ["caught", 'caught and bowled', 'bowled', 'stumped','lbw', 'hit wicket'] else 0)
bowlerdf["BowlerRun"] = bowlerdf.extra_type.apply(lambda x: 0 if x in ["legbyes", "byes"] else 1) * bowlerdf["total_run"]
bowler = bowlerdf.groupby(["ID", "bowler"], as_index=False)[["IsBowlerWicket", "BowlerRun"]].sum()
bowler["BowlingFigure"] = bowler[["IsBowlerWicket", "BowlerRun"]].apply(lambda x: '/'.join(map(str, x.values)), axis = 1)
bowler.rename(columns = {"bowler": "PlayerOfMatch"}, inplace = True)
bowler.head()

In [ ]:
batter.merge(bowler, on=["ID", "PlayerOfMatch"], how = "outer").drop(columns = ["batsman_run", "IsBatsmanBall", "IsBowlerWicket", "BowlerRun"]).head()